# CineSen — Train ABSA (standalone cho Kaggle)

Notebook **một file**, tương đương pipeline **Notebook 04** (`04_Advanced_ABSA_Modeling.ipynb`): train multi-label ABSA (DistilRoBERTa), export `absa_eval.json`, checkpoint, và `absa_movie_profiles.json`.

## Chuẩn bị trên Kaggle

1. **Bật GPU** (Settings → Accelerator → GPU T4 x2 hoặc tương đương).
2. **Thêm Dataset** chứa hai file (cùng một thư mục, hoặc trong subfolder `absa/`). Trên Kaggle đường dẫn thường lồng sâu, ví dụ `.../kaggle/input/datasets/phanvin/data-nlp-verygood/` — notebook **tự quét đệ quy** dưới `/kaggle/input`. Nếu vẫn lỗi, set `ABSA_DATA_DIR` trỏ đúng thư mục đó.
   - `labeled_absa_auto.jsonl` — output từ `03b_ABSA_AutoLabeling.ipynb`
   - `absa_clean_reviews.csv` — output từ notebook 02 (cột `tmdb_id`, `cleaned_content`)
3. (Khuyến nghị) Thêm **Secrets** `HF_TOKEN` nếu tải model Hugging Face bị giới hạn rate.

## Đầu ra (trong `/kaggle/working/absa/`)

- `absa_eval.json` — cho notebook 05
- `artifacts/<ABSA_ARTIFACT_NAME>/` — `model.pt`, `tokenizer/`, `metadata.json`
- `absa_movie_profiles.json` — cho artifact recommender / demo

Sau khi chạy xong: **Download** thư mục `working` hoặc tạo **Output Dataset** từ Kaggle.

## Cấu hình tùy chọn (environment)

- `ABSA_MODEL_NAME` — mặc định `distilroberta-base`
- `ABSA_ARTIFACT_NAME` — mặc định `absa_distilroberta_latest`
- `ABSA_DATA_DIR` — thư mục **hoặc** đường dẫn tới file `labeled_absa_auto.jsonl` (nếu auto-detect thất bại), ví dụ `/kaggle/input/datasets/phanvin/data-nlp-verygood`
- `ABSA_OUTPUT_DIR` — mặc định `/kaggle/working/absa` trên Kaggle, `.absa_kaggle_out` khi chạy local


In [ ]:
# Gói phụ thuộc (Kaggle image thường đã có torch + transformers; cài thêm nếu thiếu)
import subprocess
import sys

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import transformers
except ImportError:
    pip_install(["transformers>=4.36", "accelerate"])

print("OK: dependencies")

In [ ]:
"""
Toàn bộ pipeline: resolve đường dẫn → train → eval → lưu artifact → build movie profiles.
"""
from __future__ import annotations

import json
import os
import random
from collections import defaultdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

# Hugging Face token (Kaggle Secrets)
try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    _tok = _sec.get_secret("HF_TOKEN")
    if _tok:
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        os.environ["HF_TOKEN"] = _tok
except Exception:
    pass

# ---------------------------------------------------------------------------
# Đường dẫn: Kaggle vs local
# ---------------------------------------------------------------------------


def find_labeled_jsonl(root: Path) -> Optional[Path]:
    cand = [
        root / "labeled_absa_auto.jsonl",
        root / "absa" / "labeled_absa_auto.jsonl",
    ]
    for c in cand:
        if c.is_file():
            return c
    return None


def find_clean_csv(root: Path) -> Optional[Path]:
    cand = [
        root / "absa_clean_reviews.csv",
        root / "absa" / "absa_clean_reviews.csv",
    ]
    for c in cand:
        if c.is_file():
            return c
    return None


def resolve_data_dir() -> Path:
    """Tìm thư mục chứa nhãn. Kaggle thường lồng sâu: /kaggle/input/datasets/user/slug/..."""
    forced = os.environ.get("ABSA_DATA_DIR", "").strip()
    if forced:
        p = Path(forced)
        if p.is_file():
            p = p.parent
        if find_labeled_jsonl(p):
            return p
        raise FileNotFoundError(
            f"ABSA_DATA_DIR={forced!r} không chứa labeled_absa_auto.jsonl "
            "(có thể trỏ tới thư mục hoặc đường dẫn tới file .jsonl)."
        )

    kg = Path("/kaggle/input")
    if kg.is_dir():
        matches = list(kg.rglob("labeled_absa_auto.jsonl"))
        if not matches:
            raise FileNotFoundError(
                "Không tìm thấy labeled_absa_auto.jsonl dưới /kaggle/input (đã quét đệ quy). "
                "Thêm Dataset hoặc set ABSA_DATA_DIR=/kaggle/input/datasets/.../thư-mục-chứa-file"
            )
        # Ưu tiên thư mục có cả absa_clean_reviews.csv
        for m in sorted(matches, key=lambda x: len(x.parts)):
            parent = m.parent
            if find_clean_csv(parent):
                return parent
        return matches[0].parent

    cwd = Path.cwd()
    for base in [cwd, cwd / "Notebook_Report", cwd.parent / "Notebook_Report"]:
        if find_labeled_jsonl(base):
            return base
    raise FileNotFoundError(
        "Chạy local: đặt cwd tại thư mục có labeled_absa_auto.jsonl hoặc set ABSA_DATA_DIR."
    )


def resolve_output_dir() -> Path:
    env = os.environ.get("ABSA_OUTPUT_DIR", "").strip()
    if env:
        return Path(env)
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working/absa")
    return Path.cwd() / ".absa_kaggle_out"


DATA_DIR = resolve_data_dir()
LABELED_JSONL = find_labeled_jsonl(DATA_DIR)
ABSA_CLEAN_CSV = find_clean_csv(DATA_DIR)
OUT_DIR = resolve_output_dir()
OUT_DIR.mkdir(parents=True, exist_ok=True)

if ABSA_CLEAN_CSV is None:
    raise FileNotFoundError(
        f"Không thấy absa_clean_reviews.csv trong dataset (đã tìm trong {DATA_DIR})."
    )

print("DATA_DIR:", DATA_DIR)
print("LABELED_JSONL:", LABELED_JSONL)
print("ABSA_CLEAN_CSV:", ABSA_CLEAN_CSV)
print("OUT_DIR:", OUT_DIR)

# ---------------------------------------------------------------------------
# Schema & hyperparameters (giống notebook 04)
# ---------------------------------------------------------------------------
ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]
NUM_LABELS = len(ASPECTS) * len(SENTIMENTS)


def get_label_index(aspect: str, sentiment: str) -> int:
    if aspect not in ASPECTS or sentiment not in SENTIMENTS:
        raise ValueError(f"Unknown aspect={aspect!r} or sentiment={sentiment!r}")
    return ASPECTS.index(aspect) * len(SENTIMENTS) + SENTIMENTS.index(sentiment)


SEED = 42
MODEL_NAME = os.environ.get("ABSA_MODEL_NAME", "distilroberta-base")
ARTIFACT_SUBDIR = os.environ.get("ABSA_ARTIFACT_NAME", "absa_distilroberta_latest")

LIMIT_TOTAL: Optional[int] = None  # đặt số nhỏ để debug
VAL_RATIO = 0.2
EPOCHS = 5
BATCH_SIZE = int(os.environ.get("ABSA_BATCH_SIZE", "8"))
LR = 2e-5
MAX_LENGTH = 256
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_jsonl(path: Path, limit: Optional[int] = None):
    out = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            text = ex.get("text", "")
            labels = ex.get("labels", [])
            if text and isinstance(labels, list):
                out.append({"text": text, "labels": labels})
    return out


examples = load_jsonl(LABELED_JSONL, limit=LIMIT_TOTAL)
if not examples:
    raise RuntimeError(f"File nhãn rỗng/không hợp lệ: {LABELED_JSONL}")

rng = random.Random(SEED)
rng.shuffle(examples)

val_size = max(1, int(round(VAL_RATIO * len(examples))))
val_examples = examples[:val_size]
train_examples = examples[val_size:]

print(f"Loaded: total={len(examples)} | train={len(train_examples)} | val={len(val_examples)}")


def vectorize_labels(label_list: list) -> np.ndarray:
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    for l in label_list:
        try:
            idx = get_label_index(l["aspect"], l["sentiment"])
            vec[idx] = 1.0
        except Exception:
            continue
    return vec


class AbsaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.samples = []
        for item in data:
            self.samples.append((item["text"], vectorize_labels(item.get("labels", []))))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        text, vec = self.samples[i]
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(vec, dtype=torch.float32),
        }


class AbsaClassifier(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.backbone.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)


def predict(model: nn.Module, loader: DataLoader, threshold: float, device: str):
    model.eval()
    ys, yh = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)
            ys.append(labels)
            yh.append(preds)
    return np.vstack(ys), np.vstack(yh)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = AbsaDataset(train_examples, tokenizer, max_length=MAX_LENGTH)
val_ds = AbsaDataset(val_examples, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = AbsaClassifier(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

total_steps = max(1, len(train_loader) * EPOCHS)
warmup_steps = max(1, int(WARMUP_RATIO * total_steps))
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print("--- Train ABSA (DistilRoBERTa + warmup + grad clip) ---")
print("device:", device)
for epoch in range(EPOCHS):
    model.train()
    losses = []
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        losses.append(float(loss.item()))

    y_true, y_pred = predict(model, val_loader, threshold=THRESHOLD, device=device)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(
        f"Epoch {epoch + 1}/{EPOCHS} | loss={np.mean(losses):.4f} | "
        f"val_microF1={micro:.3f} | val_macroF1={macro:.3f}"
    )

print("Xong train.")

# Eval cuối
y_true, y_pred = predict(model, val_loader, threshold=THRESHOLD, device=device)
micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

per_label = []
for a in ASPECTS:
    for s in SENTIMENTS:
        idx = get_label_index(a, s)
        f1 = f1_score(y_true[:, idx], y_pred[:, idx], average="binary", zero_division=0)
        per_label.append({"aspect": a, "sentiment": s, "f1": float(f1)})

idx_overall = [get_label_index("overall", s) for s in SENTIMENTS]
true_cls = np.argmax(y_true[:, idx_overall], axis=1)
pred_cls = np.argmax(y_pred[:, idx_overall], axis=1)
cm3 = np.zeros((3, 3), dtype=int)
for t, p in zip(true_cls, pred_cls):
    cm3[int(t), int(p)] += 1

label_names = [(a, s) for a in ASPECTS for s in SENTIMENTS]
sample_rows = []
for i in range(min(8, len(val_examples))):
    text = val_examples[i]["text"]
    tlabs = [f"{a}:{s}" for (a, s), v in zip(label_names, y_true[i]) if v >= 0.5]
    plabs = [f"{a}:{s}" for (a, s), v in zip(label_names, y_pred[i]) if v >= 0.5]
    sample_rows.append({"text": text[:300], "true_labels": tlabs[:12], "pred_labels": plabs[:12]})

out_path = OUT_DIR / "absa_eval.json"
payload = {
    "model": MODEL_NAME,
    "n_total": int(len(examples)),
    "n_train": int(len(train_examples)),
    "n_val": int(len(val_examples)),
    "micro_f1": float(micro),
    "macro_f1": float(macro),
    "overall_sentiment_cm": cm3.tolist(),
    "sentiments": SENTIMENTS,
    "per_label_f1": per_label,
    "samples": sample_rows,
}
with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"Đã lưu -> {out_path} | microF1={micro:.3f} macroF1={macro:.3f}")

artifact_dir = OUT_DIR / "artifacts" / ARTIFACT_SUBDIR
artifact_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = artifact_dir / "model.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "num_labels": int(NUM_LABELS),
        "aspects": ASPECTS,
        "sentiments": SENTIMENTS,
        "max_length": int(MAX_LENGTH),
        "threshold": float(THRESHOLD),
        "val_micro_f1": float(micro),
        "val_macro_f1": float(macro),
    },
    ckpt_path,
)
try:
    tokenizer.save_pretrained(artifact_dir / "tokenizer")
except Exception as e:
    print("Cảnh báo tokenizer:", e)

meta_path = artifact_dir / "metadata.json"
with meta_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "schema": {"aspects": ASPECTS, "sentiments": SENTIMENTS},
            "train": {
                "seed": int(SEED),
                "val_ratio": float(VAL_RATIO),
                "epochs": int(EPOCHS),
                "batch_size": int(BATCH_SIZE),
                "lr": float(LR),
                "weight_decay": float(WEIGHT_DECAY),
                "warmup_ratio": float(WARMUP_RATIO),
                "grad_clip": float(GRAD_CLIP),
                "max_length": int(MAX_LENGTH),
                "threshold": float(THRESHOLD),
                "artifact_subdir": str(ARTIFACT_SUBDIR),
            },
            "data": {
                "n_total": int(len(examples)),
                "n_train": int(len(train_examples)),
                "n_val": int(len(val_examples)),
            },
            "val": {"micro_f1": float(micro), "macro_f1": float(macro)},
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
print(f"Checkpoint -> {ckpt_path}")

# Movie profiles
df_absa = pd.read_csv(ABSA_CLEAN_CSV).fillna("")
if "tmdb_id" not in df_absa.columns or "cleaned_content" not in df_absa.columns:
    raise ValueError("absa_clean_reviews.csv cần cột tmdb_id và cleaned_content")

MAX_REVIEWS_PER_MOVIE = 5
rows = (
    df_absa[df_absa["cleaned_content"].astype(str).str.len() >= 15]
    .groupby("tmdb_id")
    .head(MAX_REVIEWS_PER_MOVIE)
    .reset_index(drop=True)
)
print("Rows for profile inference:", len(rows))

acc_sum = defaultdict(lambda: np.zeros(NUM_LABELS, dtype=np.float32))
acc_cnt = defaultdict(int)
INF_BATCH = 32
model.eval()
texts = rows["cleaned_content"].astype(str).tolist()
movie_ids = rows["tmdb_id"].tolist()

with torch.no_grad():
    for start in range(0, len(texts), INF_BATCH):
        batch_text = texts[start : start + INF_BATCH]
        batch_mid = movie_ids[start : start + INF_BATCH]
        enc = tokenizer(
            batch_text,
            max_length=MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        for mid, p in zip(batch_mid, probs):
            acc_sum[str(mid)] += p.astype(np.float32)
            acc_cnt[str(mid)] += 1

movie_profiles = {}
for mid, vec_sum in acc_sum.items():
    cnt = acc_cnt[mid]
    if cnt <= 0:
        continue
    avg = (vec_sum / float(cnt)).tolist()
    profile = {a: {s: 0.0 for s in SENTIMENTS} for a in ASPECTS}
    for (a, s), score in zip(label_names, avg):
        profile[a][s] = float(round(score, 4))
    movie_profiles[mid] = {"n_reviews": int(cnt), "scores": profile}

out_profiles = OUT_DIR / "absa_movie_profiles.json"
with out_profiles.open("w", encoding="utf-8") as f:
    json.dump(movie_profiles, f, ensure_ascii=False, indent=2)
print(f"Profiles -> {out_profiles} (movies={len(movie_profiles)})")
print("DONE. Download /kaggle/working/absa hoặc tạo Dataset từ Output.")